In [0]:
%pip install phonenumbers pycountry rapidfuzz

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql import Window


In [0]:
#Lecture Bronze
df = spark.table("workspace.final_project.bronze_orders") # A replacer par silver clean par suppliers quand existant

# Lecture Silver
df_suppliers = spark.table("workspace.final_project.bronze_suppliers") # A remplacer par silver quand existant


In [0]:
### ORDER_ID & SUPPLIER_ID TO INT

from pyspark.sql import functions as F

from pyspark.sql import functions as F

def clean_integer_column(df, col_name, to_delete=False, accept_float=False):
    """
    Clean a column expected to contain integers.

    Behaviors:
    - If accept_float=False:
        Accept only:
        - 123
        - 123.0
        - 123.00
        Invalid values become null.
    - If accept_float=True:
        Accept any numeric value:
        - 123
        - 123.0
        - 123.45
        - 123.99
        Values are rounded to nearest integer.

    Parameters:
        df : Spark DataFrame
        col_name : str
            Column to clean
        to_delete : bool
            - False: keep invalid values as null in the cleaned column
            - True: remove rows where cleaned value is null and return them in quarantine
        accept_float : bool
            - False: only accept integers or floats ending in .0 / .00 / ...
            - True: accept any numeric float and round to integer

    Returns:
        If to_delete=False:
            df_cleaned

        If to_delete=True:
            df_valid, df_quarantine
    """

    col_str = F.trim(F.col(col_name).cast("string"))

    if accept_float:
        # Accept any numeric value (integer or float), then round to integer
        cleaned_col = F.when(
            col_str.rlike(r"^[0-9]+(\.[0-9]+)?$"),
            F.round(col_str.cast("double")).cast("int")
        ).otherwise(None)

    else:
        # Accept only integers or values ending with .0 / .00 / ...
        clean_str = F.regexp_replace(col_str, r"\.0+$", "")
        cleaned_col = F.when(
            col_str.rlike(r"^[0-9]+(\.0+)?$"),
            clean_str.cast("int")
        ).otherwise(None)

    # Add cleaned version temporarily
    df_with_clean = df.withColumn("__cleaned_col__", cleaned_col)

    if not to_delete:
        # Replace original column with cleaned column
        df_cleaned = (
            df_with_clean
            .drop(col_name)
            .withColumnRenamed("__cleaned_col__", col_name)
        )
        return df_cleaned

    # Split valid / quarantine
    df_valid = (
        df_with_clean
        .filter(F.col("__cleaned_col__").isNotNull())
        .drop(col_name)
        .withColumnRenamed("__cleaned_col__", col_name)
    )

    df_quarantine = (
        df_with_clean
        .filter(F.col("__cleaned_col__").isNull())
        .drop("__cleaned_col__")
        .withColumn(
            "quarantine_reason",
            F.when(F.col(col_name).isNull(), F.lit(f"missing_{col_name}"))
             .otherwise(F.lit(f"invalid_{col_name}"))
        )
        .withColumn("quarantine_timestamp", F.current_timestamp())
    )

    return df_valid, df_quarantine

df_int, df_quarantine = clean_integer_column(df, "order_id", to_delete=True)
df_int = clean_integer_column(df_int, "supplier_id", to_delete=False)
display(df_int)

In [0]:
# ---------- Date parsing ----------

def parse_date_col(col_name):
    c = F.trim(F.col(col_name).cast("string"))

    return F.coalesce(
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-d"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-d"))),

        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/MM/dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/M/d"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/M/dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/MM/d"))),

        F.to_date(F.try_to_timestamp(c, F.lit("dd-MM-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("d-M-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("d-MM-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("dd-M-yyyy"))),

        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-d'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-d'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-dd'T'HH:mm:ss")))
    )


# ---------- Clean date columns ----------

df_int_clean = (
    df_int
    .withColumn("order_date", parse_date_col("order_date"))
    .withColumn("delivery_date_expected", parse_date_col("delivery_date_expected"))
    .withColumn("delivery_date_actual", parse_date_col("delivery_date_actual"))
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def quarantine_full_duplicates(df, quarantine_reason="full_duplicated_line"):
    """
    Remove fully duplicated rows while keeping the first occurrence.
    Send removed duplicate rows to quarantine.

    Parameters:
        df : Spark DataFrame
        quarantine_reason : str, reason added to quarantined rows

    Returns:
        df_without_duplicates : DataFrame with only first occurrences kept
        df_quarantine : DataFrame containing removed full duplicates with quarantine metadata
    """

    window_all = Window.partitionBy(df.columns).orderBy(F.lit(1))

    df_with_rownum = df.withColumn("row_num", F.row_number().over(window_all))

    df_without_duplicates = (
        df_with_rownum
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    df_quarantine = (
        df_with_rownum
        .filter(F.col("row_num") > 1)
        .drop("row_num")
        .withColumn("quarantine_reason", F.lit(quarantine_reason))
        .withColumn("quarantine_timestamp", F.current_timestamp())
    )

    return df_without_duplicates, df_quarantine

df_without_duplicates, df_quarantine = quarantine_full_duplicates(df_int_clean, quarantine_reason="full_duplicated_line")

In [0]:
"""

# Define df_valid with cleaned order_id and supplier_id
# Keep only rows where order_id and supplier_id are not null and order_id is not an invalid string
df_valid = df_first_occurrences.filter(
    (F.col("order_id").isNotNull()) &         # real null check
    (F.trim(F.col("order_id")) != "") &       # remove empty strings
    (F.lower(F.col("order_id")) != "null") &  # remove string "null"
    (F.lower(F.col("order_id")) != "none") &  # remove string "none"
    (F.col("supplier_id").isNotNull())        # supplier_id must not be null
)
"""

In [0]:
### VERIFICATION COHERENCE DATES

from pyspark.sql import functions as F

def validate_date_order(df, date_late_col, date_early_col, primary_key):
    """
    Filter DataFrame rows where date_late_col >= date_early_col (if neither is null).
    Rows failing this condition are moved to a quarantine DataFrame with reason and timestamp.

    Parameters:
        df : Spark DataFrame
        date_late_col : str, the column that should be later (delivery_date_expected)
        date_early_col : str, the column that should be earlier (order_date)

    Returns:
        df_valid : DataFrame with valid rows
        df_quarantine : DataFrame with invalid rows and quarantine columns
    """

    # Step 1: valid rows
    df_valid = df.filter(
        (F.col(date_late_col).isNull()) | 
        (F.col(date_early_col).isNull()) |
        (F.col(date_late_col) >= F.col(date_early_col))
    )

    # Step 2: rows failing the condition
    df_quarantine = df.join(
        df_valid.select(primary_key),  # assuming order_id is unique
        on=primary_key,
        how="left_anti"
    ).withColumn(
        "quarantine_reason",
        F.lit(f"{date_late_col}_before_{date_early_col}")
    ).withColumn(
        "quarantine_timestamp",
        F.current_timestamp()
    )

    return df_valid, df_quarantine

# Verification delivery_date_expected > order_date
df_time_coherence, df_quarantine_time= validate_date_order(df_without_duplicates,
                                                              date_late_col="delivery_date_expected",
                                                              date_early_col="order_date",
                                                              primary_key="order_id")

# Verification delivery_date_actual > order_date
df_time_coherence, df_quarantine_time_2 = validate_date_order(df_without_duplicates,
                                                              date_late_col="delivery_date_actual",
                                                              date_early_col="order_date",
                                                              primary_key="order_id")



In [0]:
### VERIFICATION COHERENCE SUPPLIER_ID

from pyspark.sql import functions as F

def validate_foreign_key(df_main, df_reference, fk_col, pk_col, primary_key_main):
    """
    Validate a foreign key column in a main DataFrame against a reference DataFrame.
    Rows with invalid foreign keys are moved to a quarantine DataFrame with reason and timestamp.

    Parameters:
        df_main : Spark DataFrame containing the main table
        df_reference : Spark DataFrame containing the reference table (primary keys)
        fk_col : str, column name in df_main that is the foreign key
        pk_col : str, column name in df_reference that is the primary key
        primary_key_main : str, column name of the primary key in df_main (unique per row)

    Returns:
        df_valid : rows in df_main with fk_col valid
        df_quarantine : rows in df_main with invalid fk_col, plus quarantine_reason and quarantine_timestamp
    """

    # Step 1: keep only rows with valid foreign keys
    df_valid = df_main.join(
        df_reference.select(pk_col).distinct().withColumnRenamed(pk_col, fk_col),
        on=fk_col,
        how="inner"
    )

    # Step 2: rows with invalid foreign keys
    df_quarantine = df_main.join(
        df_valid.select(primary_key_main),
        on=primary_key_main,
        how="left_anti"
    ).withColumn(
        "quarantine_reason",
        F.lit(f"{fk_col}_not_in_reference")
    ).withColumn(
        "quarantine_timestamp",
        F.current_timestamp()
    )

    return df_valid, df_quarantine

df_valid, df_orders_quarantine = validate_foreign_key(df_main=df_time_coherence,
                                                             df_reference=df_suppliers,
                                                             fk_col="supplier_id",
                                                             pk_col="supplier_id",
                                                             primary_key_main="order_id"
                                                             )

In [0]:

### TABLE SPLIT

# Split of items and orders dataset
# Select columns for orders
df_orders = df_valid.select(
    "order_id",
    "supplier_id",
    "order_date",
    "delivery_date_expected",
    "delivery_date_actual"
)

# Select columns for items
df_items = df_valid.select(
    "order_id",
    F.posexplode("items").alias("item_index", "item")
)

# Flatten items
df_items = df_items.select(
    "order_id",
    "item_index",
    "item.product_category",
    "item.quantity",
    "item.unit_price"
)

# Add item_id
# We add a unique identifier for each item with hash 
df_items = df_items.withColumn(
    "item_id",
    F.sha2(F.concat_ws("||", F.col("order_id"), F.col("item_index")), 256)
)

In [0]:
###

# Gérer les doublons
df_items_dedup, df_quarantine = quarantine_full_duplicates(df_items, quarantine_reason="full_duplicated_line")

def missing_values_report(df):
    """
    Return a long-format DataFrame with missing value counts per column.

    Missing values include:
    - true null
    - empty string ""
    - string values: "nan", "none", "null" (case insensitive)

    Parameters:
        df : Spark DataFrame

    Returns:
        df_missing : Spark DataFrame with columns:
            - column_name
            - missing_count
    """

    # Build one small DataFrame per column, then union all
    dfs = []

    for c in df.columns:
        df_col = df.select(
            F.lit(c).alias("column_name"),
            F.sum(
                F.when(
                    F.col(c).isNull() |
                    (F.trim(F.col(c).cast("string")) == "") |
                    (F.lower(F.trim(F.col(c).cast("string"))).isin("nan", "none", "null")),
                    1
                ).otherwise(0)
            ).alias("missing_count")
        )
        dfs.append(df_col)

    # Union all per-column results
    df_missing = dfs[0]
    for d in dfs[1:]:
        df_missing = df_missing.unionByName(d)

    return df_missing

df_missing_items = missing_values_report(df_items_dedup)

display(df_missing_items)
# Gerer les missing data (au moins retour statistique)
# Gerer le type de quantity (mettre en int)
# Ou gérer les valeurs aberrantes ?


In [0]:
from pyspark.sql import functions as F

# =========================================
# 1. DROP FULL DUPLICATE ROWS
# =========================================

# Keep only the first occurrence of fully duplicated rows
df_items_dedup = df_items.dropDuplicates()

# Count number of removed full duplicate rows
full_duplicate_count = df_items.count() - df_items_dedup.count()

print(f"Full duplicate rows removed: {full_duplicate_count}")


# =========================================
# 2. COUNT MISSING VALUES BY COLUMN
#    (null, empty string, 'nan', 'none', 'null')
# =========================================

missing_counts_expr = []

for c in df_items_dedup.columns:
    missing_expr = F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == "") |
            (F.lower(F.trim(F.col(c).cast("string"))).isin("nan", "none", "null")),
            1
        ).otherwise(0)
    ).alias(c)
    
    missing_counts_expr.append(missing_expr)

df_missing_counts = df_items_dedup.select(missing_counts_expr)

print("Missing values count by column:")
display(df_missing_counts)


# =========================================
# 3. CLEAN QUANTITY COLUMN -> INT
#    Valid: 123, 123.0, 123.00
#    Invalid: replaced by null
# =========================================

quantity_str = F.col("quantity").cast("string")
quantity_clean_str = F.regexp_replace(quantity_str, r"\.0+$", "")

df_items_clean = df_items_dedup.withColumn(
    "quantity",
    F.when(
        quantity_str.rlike(r"^[0-9]+(\.0+)?$"),
        quantity_clean_str.cast("int")
    ).otherwise(None)
)

# Optional: count invalid quantity values after cleaning
invalid_quantity_count = df_items_clean.filter(F.col("quantity").isNull()).count()

print(f"Rows where quantity is null after cleaning: {invalid_quantity_count}")

display(df_items_clean)